# F5-TTS Russian Audiobook Generator
Generates high-quality Russian audio using F5-TTS-Russian with voice cloning.

**Runtime: GPU (T4 or better)**

Audio saves to Google Drive (`/My Drive/tts_output/`) so it survives disconnects.

In [ ]:
#@title 1. Setup: mount Drive, clone repo, install deps, download model
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/My Drive/tts_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

!git clone https://github.com/stuk88/post-scarcity-architecture.git repo
!pip install -q f5-tts soundfile

from huggingface_hub import hf_hub_download
os.makedirs('model', exist_ok=True)
for f in ['model_last.safetensors', 'vocab.txt']:
    hf_hub_download('hotstone228/F5-TTS-Russian', f, local_dir='model')
print('Setup complete!')

In [ ]:
#@title 2. Generate all chapters (saves each WAV to Google Drive)
import re, time, os
import numpy as np
import soundfile as sf
from pathlib import Path
from f5_tts.model import DiT
from f5_tts.infer.utils_infer import (
    load_model, load_vocoder, preprocess_ref_audio_text, infer_process
)

SPEED = 0.9
NFE_STEP = 16
SR = 24000
CHAPTER_DIR = Path('repo/ru_tts_translation/chapter_texts')
REF_AUDIO = 'repo/ru_tts_translation/ref_audio.wav'
OUTPUT = Path('/content/drive/My Drive/tts_output')

print('Loading vocoder...')
vocoder = load_vocoder(vocoder_name='vocos', is_local=False)

print('Loading F5-TTS-Russian...')
model = load_model(
    DiT,
    dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4),
    'model/model_last.safetensors',
    mel_spec_type='vocos',
    vocab_file='model/vocab.txt',
)

print('Processing reference audio...')
ref_audio, ref_text = preprocess_ref_audio_text(REF_AUDIO, '')
print(f'Ref text: {ref_text[:80]}...')

chapter_files = sorted(CHAPTER_DIR.glob('*.txt'))
print(f'\nGenerating {len(chapter_files)} chapters (saving to Google Drive)...\n')

t0 = time.time()
wav_files = []

for i, ch_file in enumerate(chapter_files):
    out_wav = OUTPUT / f'{ch_file.stem}.wav'
    if out_wav.exists():
        print(f'  [skip] {out_wav.name}')
        wav_files.append(out_wav)
        continue

    text = ch_file.read_text(encoding='utf-8').strip()
    print(f'  [{i+1}/{len(chapter_files)}] {ch_file.name} ({len(text)} chars)')

    segments = [s.strip() for s in text.split('\n\n') if s.strip()]
    audio_parts = []

    for seg in segments:
        dots_only = seg.replace('.', '').replace(' ', '')
        if not dots_only:
            pause = min(seg.count('.') * 0.4, 3.0)
            audio_parts.append(np.zeros(int(SR * pause), dtype=np.float32))
            continue

        sentences = re.split(r'(?<=[.!?])\s+', seg)
        chunks, current = [], ''
        for s in sentences:
            if len(current) + len(s) > 500 and current:
                chunks.append(current.strip())
                current = s
            else:
                current = f'{current} {s}' if current else s
        if current.strip():
            chunks.append(current.strip())

        for chunk in chunks:
            try:
                audio, sr, _ = infer_process(
                    ref_audio, ref_text, chunk,
                    model, vocoder,
                    speed=SPEED, nfe_step=NFE_STEP,
                )
                audio_parts.append(audio.astype(np.float32))
            except Exception as e:
                print(f'    ERR: {str(e)[:80]}')

        audio_parts.append(np.zeros(int(SR * 0.5), dtype=np.float32))

    if audio_parts:
        full = np.concatenate(audio_parts)
        sf.write(str(out_wav), full, SR)
        dur = len(full) / SR
        elapsed = time.time() - t0
        print(f'    -> {out_wav.name} ({dur:.0f}s audio) [{elapsed:.0f}s elapsed]')
        wav_files.append(out_wav)

print(f'\nDone! {len(wav_files)} chapters in {time.time()-t0:.0f}s')
print(f'Files saved to Google Drive: tts_output/')

In [ ]:
#@title 3. Combine into MP3 (also saved to Drive)
import subprocess

OUTPUT = Path('/content/drive/My Drive/tts_output')
wav_files = sorted(OUTPUT.glob('*.wav'))
print(f'Combining {len(wav_files)} chapters...')

with open('/tmp/filelist.txt', 'w') as f:
    for wav in wav_files:
        f.write(f"file '{wav}'\n")

mp3_path = OUTPUT / 'Post_Scarcity_v3_ru.mp3'
subprocess.run([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
    '-i', '/tmp/filelist.txt',
    '-codec:a', 'libmp3lame', '-qscale:a', '2', '-ar', '24000',
    str(mp3_path)
], capture_output=True)

size = mp3_path.stat().st_size / (1024*1024)
print(f'Saved: {mp3_path} ({size:.1f} MB)')
print('File is in your Google Drive under tts_output/')